In [0]:


WITH cleaned_users AS (

    -- Clean the user table and create grouping fields age_group,province,race_group,province_group,
    SELECT
        COALESCE(CAST(UserID AS STRING), '0') AS user_id,
        COALESCE(Name, 'No Name') AS name,
        COALESCE(Surname, 'No Surname') AS surname,
        COALESCE(Gender, 'No Gender') AS gender,
        COALESCE(Race, 'No Race') AS race,
        COALESCE(CAST(ROUND(Age, 0) AS INT), 0) AS age,
        COALESCE(Province, 'No Province') AS province,

        CASE
            WHEN Age IS NULL THEN 'Unknown'
            WHEN Age < 18 THEN 'Under 18'
            WHEN Age BETWEEN 18 AND 24 THEN '18-24'
            WHEN Age BETWEEN 25 AND 34 THEN '25-34'
            WHEN Age BETWEEN 35 AND 44 THEN '35-44'
            WHEN Age BETWEEN 45 AND 54 THEN '45-54'
            ELSE '55+'
        END AS age_group,

        CASE
            WHEN Province IN ('Gauteng', 'Western Cape', 'KwaZulu-Natal', 'Kwazulu Natal') THEN 'Top Provinces'
            WHEN Province IS NULL THEN 'Unknown'
            ELSE 'Other Provinces'
        END AS province_group,

        CASE
            WHEN LOWER(Race) IN ('black', 'white', 'coloured', 'indian_asian', 'indian') THEN LOWER(Race)
            WHEN Race IS NULL THEN 'Unknown'
            ELSE 'Other'
        END AS race_group

    FROM workspace.default.bright_tv_user),

cleaned_viewership AS (

    -- Clean the session table and convert UTC to South African time
    SELECT
        COALESCE(CAST(UserID AS STRING), '0') AS user_id,
        COALESCE(Channel2, 'No Channel') AS channel,
        RecordDate2 AS recorddate_raw,

        -- Original UTC timestamp
        TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm') AS record_timestamp_utc,

        -- Converted SA timestamp
        FROM_UTC_TIMESTAMP(
            TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
            'Africa/Johannesburg'
        ) AS record_timestamp_sa,

        -- Separate SA date and time
        CAST(
            FROM_UTC_TIMESTAMP(
                TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                'Africa/Johannesburg'
            ) AS DATE
        ) AS record_date,

        DATE_FORMAT(
            FROM_UTC_TIMESTAMP(
                TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                'Africa/Johannesburg'
            ),
            'HH:mm:ss'
        ) AS record_time,

        HOUR(
            FROM_UTC_TIMESTAMP(
                TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                'Africa/Johannesburg'
            )
        ) AS viewing_hour,

        DATE_TRUNC(
            'month',
            CAST(
                FROM_UTC_TIMESTAMP(
                    TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                    'Africa/Johannesburg'
                ) AS DATE
            )
        ) AS activity_month,

        DATE_FORMAT(
            CAST(
                FROM_UTC_TIMESTAMP(
                    TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                    'Africa/Johannesburg'
                ) AS DATE), 'EEEE'
        ) AS day_name,

        CASE
            WHEN DAYOFWEEK(
                CAST(
                    FROM_UTC_TIMESTAMP(
                        TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                        'Africa/Johannesburg'
                    ) AS DATE)
            ) IN (1, 7) THEN 'Weekend'
            ELSE 'Weekday'
        END AS week_part,

        CASE
            WHEN HOUR(
                FROM_UTC_TIMESTAMP(
                    TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                    'Africa/Johannesburg')
            ) BETWEEN 5 AND 11 THEN 'Morning'
            WHEN HOUR(
                FROM_UTC_TIMESTAMP(
                    TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                    'Africa/Johannesburg')
            ) BETWEEN 12 AND 16 THEN 'Afternoon'
            WHEN HOUR(
                FROM_UTC_TIMESTAMP(
                    TO_TIMESTAMP(RecordDate2, 'yyyy/MM/dd HH:mm'),
                    'Africa/Johannesburg')
            ) BETWEEN 17 AND 20 THEN 'Evening'
            ELSE 'Night'
        END AS time_bucket,

        -- Duration converted to seconds
        COALESCE(
            HOUR(CAST(`Duration 2` AS TIMESTAMP)) * 3600 +
            MINUTE(CAST(`Duration 2` AS TIMESTAMP)) * 60 +
            SECOND(CAST(`Duration 2` AS TIMESTAMP)),
            0) AS duration_seconds

    FROM workspace.default.bright_tv_viewers),

base AS (

    -- Join user profiles to every viewing session
    SELECT
        u.user_id,
        u.name,
        u.surname,
        u.gender,
        u.race,
        u.age,
        u.age_group,
        u.province,
        u.province_group,
        u.race_group,
        v.channel,
        v.recorddate_raw,
        v.record_timestamp_utc,
        v.record_timestamp_sa,
        v.record_date,
        v.record_time,
        v.viewing_hour,
        v.activity_month,
        v.day_name,
        v.week_part,
        v.time_bucket,
        v.duration_seconds
    FROM cleaned_users u
    LEFT JOIN cleaned_viewership v
        ON u.user_id = v.user_id),

daily_usage AS (

    -- Daily usage table to identify low-consumption days
    SELECT
        record_date,
        COUNT(*) AS daily_sessions,
        COUNT(DISTINCT user_id) AS daily_active_users,
        SUM(duration_seconds) AS daily_watch_duration
    FROM base
    WHERE record_date IS NOT NULL
      AND channel <> 'No Channel'
    GROUP BY record_date),

daily_usage_flagged AS (

    -- Flag low-consumption days using average daily sessions
    SELECT
        d.*,
        AVG(daily_sessions) OVER () AS avg_daily_sessions,
        CASE
            WHEN daily_sessions < AVG(daily_sessions) OVER () THEN 'Low Consumption Day'
            ELSE 'Normal/High Consumption Day'
        END AS day_consumption_flag
    FROM daily_usage d),

user_stats AS (

    -- User-level engagement metrics
    SELECT
        user_id,
        COUNT(CASE WHEN channel <> 'No Channel' THEN 1 END) AS total_views,
        COUNT(DISTINCT CASE WHEN channel <> 'No Channel' THEN channel END) AS channels_watched,
        COUNT(DISTINCT record_date) AS active_days,
        MIN(record_date) AS first_watch_date,
        MAX(record_date) AS last_watch_date,
        SUM(COALESCE(duration_seconds, 0)) AS total_watch_duration_seconds,
        AVG(CASE WHEN channel <> 'No Channel' THEN duration_seconds END) AS avg_watch_duration_seconds,

        SUM(CASE WHEN week_part = 'Weekday' AND channel <> 'No Channel' THEN 1 ELSE 0 END) AS weekday_views,
        SUM(CASE WHEN week_part = 'Weekend' AND channel <> 'No Channel' THEN 1 ELSE 0 END) AS weekend_views,

        SUM(CASE WHEN time_bucket = 'Morning' AND channel <> 'No Channel' THEN 1 ELSE 0 END) AS morning_views,
        SUM(CASE WHEN time_bucket = 'Afternoon' AND channel <> 'No Channel' THEN 1 ELSE 0 END) AS afternoon_views,
        SUM(CASE WHEN time_bucket = 'Evening' AND channel <> 'No Channel' THEN 1 ELSE 0 END) AS evening_views,
        SUM(CASE WHEN time_bucket = 'Night' AND channel <> 'No Channel' THEN 1 ELSE 0 END) AS night_views
    FROM base
    GROUP BY user_id),

fav_channel AS (

    -- Most watched channel per user
    SELECT user_id, channel AS favorite_channel
    FROM (
        SELECT
            user_id,
            channel,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY COUNT(*) DESC, channel
            ) AS rn
        FROM base
        WHERE channel IS NOT NULL
          AND channel <> 'No Channel'
        GROUP BY user_id, channel
    ) x
    WHERE rn = 1),

peak_day AS (

    -- Most active viewing date per user
    SELECT user_id, record_date AS peak_viewing_day
    FROM (
        SELECT
            user_id,
            record_date,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY COUNT(*) DESC, record_date
            ) AS rn
        FROM base
        WHERE record_date IS NOT NULL
          AND channel <> 'No Channel'
        GROUP BY user_id, record_date) x
    WHERE rn = 1),

preferred_day AS (

    -- Most common day name per user
    SELECT user_id, day_name AS preferred_day_name
    FROM (
        SELECT
            user_id,
            day_name,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY COUNT(*) DESC, day_name
            ) AS rn
        FROM base
        WHERE day_name IS NOT NULL
          AND channel <> 'No Channel'
        GROUP BY user_id, day_name) x
    WHERE rn = 1),

time_pref AS (

    -- Preferred time of day per user
    SELECT user_id, time_bucket AS preferred_time_of_day
    FROM (
        SELECT
            user_id,
            time_bucket,
            COUNT(*) AS cnt,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY COUNT(*) DESC, time_bucket
            ) AS rn
        FROM base
        WHERE time_bucket IS NOT NULL
          AND channel <> 'No Channel'
        GROUP BY user_id, time_bucket) x
    WHERE rn = 1),

most_recent_watch AS (

    -- Most recent session per user
    SELECT
        user_id,
        record_date AS most_recent_watch_date,
        record_time AS most_recent_watch_time,
        record_timestamp_sa AS most_recent_watch_datetime,
        channel AS most_recent_channel
    FROM (
        SELECT
            user_id,
            record_date,
            record_time,
            record_timestamp_sa,
            channel,
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY record_timestamp_sa DESC
            ) AS rn
        FROM base
        WHERE record_timestamp_sa IS NOT NULL
          AND channel <> 'No Channel') x
    WHERE rn = 1),

user_master AS (

    -- One user summary row per user
    SELECT
        u.user_id,
        u.name,
        u.surname,
        u.gender,
        u.race,
        u.age,
        u.age_group,
        u.province,
        u.province_group,
        u.race_group,

        COALESCE(s.total_views, 0) AS total_views,
        COALESCE(s.channels_watched, 0) AS channels_watched,
        COALESCE(s.active_days, 0) AS active_days,
        s.first_watch_date,
        s.last_watch_date,
        COALESCE(s.total_watch_duration_seconds, 0) AS total_watch_duration_seconds,
        ROUND(COALESCE(s.avg_watch_duration_seconds, 0), 2) AS avg_watch_duration_seconds,

        COALESCE(s.weekday_views, 0) AS weekday_views,
        COALESCE(s.weekend_views, 0) AS weekend_views,
        COALESCE(s.morning_views, 0) AS morning_views,
        COALESCE(s.afternoon_views, 0) AS afternoon_views,
        COALESCE(s.evening_views, 0) AS evening_views,
        COALESCE(s.night_views, 0) AS night_views,

        COALESCE(fc.favorite_channel, 'No Activity') AS favorite_channel,
        pd.peak_viewing_day,
        COALESCE(pfd.preferred_day_name, 'No Activity') AS preferred_day_name,
        COALESCE(tp.preferred_time_of_day, 'No Activity') AS preferred_time_of_day,
        mrw.most_recent_watch_date,
        mrw.most_recent_watch_time,
        mrw.most_recent_watch_datetime,
        COALESCE(mrw.most_recent_channel, 'No Activity') AS most_recent_channel,

        ROUND(
            CASE
                WHEN COALESCE(s.active_days, 0) = 0 THEN 0
                ELSE COALESCE(s.total_views, 0) * 1.0 / s.active_days
            END, 2
        ) AS avg_views_per_active_day,

        ROUND(
            CASE
                WHEN COALESCE(s.active_days, 0) = 0 THEN 0
                ELSE COALESCE(s.total_watch_duration_seconds, 0) * 1.0 / s.active_days
            END, 2
        ) AS avg_duration_per_active_day,

        CASE
            WHEN COALESCE(s.total_views, 0) >= 50 THEN 'High'
            WHEN COALESCE(s.total_views, 0) BETWEEN 20 AND 49 THEN 'Medium'
            WHEN COALESCE(s.total_views, 0) BETWEEN 1 AND 19 THEN 'Low'
            ELSE 'No Activity'
        END AS activity_level,

        COALESCE(s.total_views, 0) * COALESCE(s.channels_watched, 0) AS engagement_score,

        CASE
            WHEN s.last_watch_date IS NULL THEN NULL
            ELSE DATEDIFF(CURRENT_DATE(), s.last_watch_date)
        END AS recency_days,

        CASE
            WHEN s.last_watch_date IS NULL THEN 'No Activity'
            WHEN DATEDIFF(CURRENT_DATE(), s.last_watch_date) <= 7 THEN 'Active'
            WHEN DATEDIFF(CURRENT_DATE(), s.last_watch_date) <= 30 THEN 'Warm'
            WHEN DATEDIFF(CURRENT_DATE(), s.last_watch_date) <= 90 THEN 'Cold'
            ELSE 'Inactive'
        END AS recency_bucket,

        CASE
            WHEN s.first_watch_date IS NULL OR s.last_watch_date IS NULL THEN 0
            ELSE DATEDIFF(s.last_watch_date, s.first_watch_date)
        END AS tenure_days,

        CASE
            WHEN COALESCE(s.weekday_views, 0) > COALESCE(s.weekend_views, 0) THEN 'Weekday Viewer'
            WHEN COALESCE(s.weekend_views, 0) > COALESCE(s.weekday_views, 0) THEN 'Weekend Viewer'
            WHEN COALESCE(s.total_views, 0) = 0 THEN 'No Activity'
            ELSE 'Balanced'
        END AS viewing_pattern

    FROM cleaned_users u
    LEFT JOIN user_stats s
        ON u.user_id = s.user_id
    LEFT JOIN fav_channel fc
        ON u.user_id = fc.user_id
    LEFT JOIN peak_day pd
        ON u.user_id = pd.user_id
    LEFT JOIN preferred_day pfd
        ON u.user_id = pfd.user_id
    LEFT JOIN time_pref tp
        ON u.user_id = tp.user_id
    LEFT JOIN most_recent_watch mrw
        ON u.user_id = mrw.user_id)

-- Final output:
-- session-level rows + user-level summary + daily-consumption flags
SELECT
    b.user_id,
    b.name,
    b.surname,
    b.gender,
    b.race,
    b.age,
    b.age_group,
    b.province,
    b.province_group,
    b.race_group,

    b.channel,
    b.recorddate_raw,
    b.record_timestamp_utc,
    b.record_timestamp_sa,
    b.record_date,
    b.record_time,
    b.viewing_hour,
    b.activity_month,
    b.day_name,
    b.week_part,
    b.time_bucket,
    b.duration_seconds,

    d.daily_sessions,
    d.daily_active_users,
    d.daily_watch_duration,
    ROUND(d.avg_daily_sessions, 2) AS avg_daily_sessions,
    d.day_consumption_flag,

    u.total_views,
    u.channels_watched,
    u.active_days,
    u.first_watch_date,
    u.last_watch_date,
    u.total_watch_duration_seconds,
    u.avg_watch_duration_seconds,
    u.weekday_views,
    u.weekend_views,
    u.morning_views,
    u.afternoon_views,
    u.evening_views,
    u.night_views,
    u.favorite_channel,
    u.peak_viewing_day,
    u.preferred_day_name,
    u.preferred_time_of_day,
    u.most_recent_watch_date,
    u.most_recent_watch_time,
    u.most_recent_watch_datetime,
    u.most_recent_channel,
    u.avg_views_per_active_day,
    u.avg_duration_per_active_day,
    u.activity_level,
    u.engagement_score,
    u.recency_days,
    u.recency_bucket,
    u.tenure_days,
    u.viewing_pattern

FROM base b
LEFT JOIN daily_usage_flagged d
    ON b.record_date = d.record_date
LEFT JOIN user_master u
    ON b.user_id = u.user_id;